In [ ]:
import os
import sys
sys.path.append('./coeqwalpackage')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

from coeqwalpackage.metrics import percent_change_from_baseline
from coeqwalpackage import cqwlutils as cu
from coeqwalpackage.plotting import (custom_parallel_coordinates_highlight_scenarios, custom_parallel_coordinates_highlight_scenarios_baseline_at_zero)
from scipy import stats

In [ ]:
CtrlFile = 'CalSim3DataExtractionInitFile_v4.xlsx'
CtrlTab = 'Init'

# init_data = cu.read_init_file(CtrlFile, CtrlTab)
# GroupDataDirPath = init_data[16]  
# ScenarioDir = init_data[17]      
(ScenarioListFile, ScenarioListTab, ScenarioListPath, DVDssNamesOutPath, SVDssNamesOutPath, ScenarioIndicesOutPath, DssDirsOutPath, VarListPath, VarListFile, VarListTab, VarOutPath, DataOutPath, ConvertDataOutPath, ExtractionSubPath, DemandDeliverySubPath, ModelSubPath, GroupDataDirPath, ScenarioDir, *_) = cu.read_init_file(CtrlFile, CtrlTab)

performance_metrics_base = os.path.join(ScenarioDir, "Performance_Metrics")
tiers_base = os.path.join(performance_metrics_base, "Tiered_Outcome_Measures")
metrics_base = os.path.join(performance_metrics_base, "Metrics")
plots_base = os.path.join(performance_metrics_base, "Plots")

os.makedirs(performance_metrics_base, exist_ok=True)
os.makedirs(metrics_base, exist_ok=True)
os.makedirs(tiers_base, exist_ok=True)
os.makedirs(plots_base, exist_ok=True)

# metrics_path = os.path.join(GroupDataDirPath, "metrics_output", "all_metrics_output.csv")
metrics_path = os.path.join(ScenarioDir, "Performance_Metrics", "Metrics", "All_Metrics", "all_metrics_output.csv")

df = pd.read_csv(metrics_path, index_col=0)
df.index.name = 'Scenario'

flood_all = pd.read_csv(os.path.join(metrics_base, "Reservoir_FloodRisk", "floodrisk_all_metrics.csv"), index_col=0)
storage_all = pd.read_csv(os.path.join(metrics_base, "Reservoir_Storage", "storage_all_metrics.csv"), index_col=0)
sal_all = pd.read_csv(os.path.join(metrics_base, "Salinity", "salinity_all_metrics.csv"), index_col=0)
gw_all = pd.read_csv(os.path.join(metrics_base, "Groundwater", "groundwater_all_metrics.csv"), index_col=0)

flood_tiers = flood_all[[c for c in flood_all.columns if 'FloodTier' in c]]
storage_tiers = storage_all[[c for c in storage_all.columns if '_Tier' in c]]
sal_tiers = sal_all[[c for c in sal_all.columns if 'Tier' in c]]
gw_tiers = gw_all[[c for c in gw_all.columns if '_Tier' in c]]
tiers_df = pd.concat([flood_tiers, storage_tiers, sal_tiers, gw_tiers], axis=1, join='outer').sort_index()

In [ ]:
FIGSIZE = (20, 7)
COLORS = ['black', 'red', 'blue', 'green', 'orange', 'purple', 'yellow', 'cyan', 'magenta']

SCENARIO_SETS = [
    {"name": "s20_s11_s21_s23_s24_s47_s51", "baseline": 20, "compare": [11, 21, 24, 23, 47, 51]},
    {"name": "s20_s25_s26_s27_s28_s62", "baseline": 20, "compare": [25, 26, 27, 28, 62]},
    {"name": "s20_s29_s30_s31_s32_s33_s46", "baseline": 20, "compare": [30, 29, 32, 31, 33, 46]},
    {"name": "s20_s39_s40_s41_s42", "baseline": 20, "compare": [40, 41, 42, 39]},
    {"name": "s20_s44_s45_s65", "baseline": 20, "compare": [44, 45, 65]}]

for s in SCENARIO_SETS:
    s["baseline_id"] = f's{s["baseline"]:04d}'
    s["compare_ids"] = [f's{c:04d}' for c in s["compare"]]
    s["all_ids"] = [s["baseline_id"]] + s["compare_ids"]
    s["colors"] = COLORS[:len(s["all_ids"])]
    s["labels"] = [f'{s["baseline_id"]} (Baseline)'] + s["compare_ids"]

METRIC_GROUPS = {
    'Storage_Apr': [c for c in df.columns if c.startswith('Apr') and 'S_' in c and 'TAF' in c],
    'Deliveries': [c for c in df.columns if 'DEL_' in c],
    'Salinity': [c for c in df.columns if 'X2_' in c or 'EC_' in c]
}
METRIC_GROUPS = {k: v for k, v in METRIC_GROUPS.items() if v}

TIER_GROUPS = {
    'Tier_FloodRisk': [c for c in tiers_df.columns if 'FloodTier' in c],
    'Tier_Storage': [c for c in tiers_df.columns if 'Storage_Tier' in c],
    'Tier_Salinity': [c for c in tiers_df.columns if 'Salinity' in c],
    'Tier_GW': [c for c in tiers_df.columns if '_Tier' in c and 'WBA' in c]}
TIER_GROUPS = {k: v for k, v in TIER_GROUPS.items() if v}

def short_label(c):
    for p in ['Apr_Avg_', 'Sep_Avg_', 'Ann_Avg_', 'Fall_Ann_Avg_', 'Spring_Ann_Avg_',
              '_TAF', '_CFS', '_KM', '_UMHOS/CM', '_FloodTier', '_Storage_Tier', '_Tier', 'GW_', 'S_']:
        c = c.replace(p, '')
    return c

print("Metric Groups:")
for k, v in METRIC_GROUPS.items():
    print(f"  {k}: {len(v)} cols")

print("\nTier Groups:")
for k, v in TIER_GROUPS.items():
    print(f"  {k}: {len(v)} cols")

## 1. Parallel Line Plots

In [ ]:
for sset in SCENARIO_SETS:    
    for grp, cols in METRIC_GROUPS.items():
        labels = [short_label(c) for c in cols]
        ideal = 'bottom' if 'Salinity' in grp else 'top'
        minmax = ['min'] * len(cols) if 'Salinity' in grp else ['max'] * len(cols)
        
        save_path = os.path.join(plots_base, f"parallel_{sset['name']}_{grp}.png")
        custom_parallel_coordinates_highlight_scenarios(
            objs=df[cols], columns_axes=cols, axis_labels=labels,
            ideal_direction=ideal, minmaxs=minmax,
            highlight_indices=sset['all_ids'], highlight_colors=sset['colors'],
            highlight_descriptions=sset['labels'],
            title=f"{sset['name']}: {grp}", fontsize=10, figsize=FIGSIZE,
            save_fig_filename=save_path)
        plt.show()
        print(f"Saved: {save_path}")

In [ ]:
for sset in SCENARIO_SETS:
    for grp, cols in METRIC_GROUPS.items():
        if sset['baseline_id'] not in df.index:
            print(f"Baseline {sset['baseline_id']} not found")
            continue
        
        pct_df = percent_change_from_baseline(df[cols], sset['baseline_id'])
        labels = [short_label(c) for c in cols]
        
        save_path = os.path.join(plots_base, f"parallel_pct_{sset['name']}_{grp}.png")
        custom_parallel_coordinates_highlight_scenarios_baseline_at_zero(
            objs=pct_df, columns_axes=cols, axis_labels=labels,
            highlight_indices=sset['all_ids'], highlight_colors=sset['colors'],
            highlight_descriptions=sset['labels'],
            title=f"{sset['name']}: {grp} — % Change from {sset['baseline_id']}",
            fontsize=10, figsize=FIGSIZE,
            save_fig_filename=save_path)
        
        plt.show()
        print(f"Saved: {save_path}")

## 2. NOD/SOD Radar and Parallel Line Plots

In [ ]:
NOD_reservoirs = ["S_SHSTA_Storage_Tier", "S_TRNTY_Storage_Tier","S_OROVL_Storage_Tier", "S_FOLSM_Storage_Tier"]

SOD_reservoirs = ["S_MELON_Storage_Tier", "S_MLRTN_Storage_Tier", "S_SLUIS_CVP_Storage_Tier", "S_SLUIS_SWP_Storage_Tier"]

NOD_gw = ["WBA2_Tier", "WBA3_Tier", "WBA4_Tier", "WBA5_Tier", "WBA6_Tier", "WBA7N_Tier", "WBA7S_Tier", "WBA8N_Tier", "WBA8S_Tier", "WBA9_Tier", "WBA10_Tier", "WBA11_Tier", "WBA12_Tier", "WBA13_Tier", "WBA14_Tier", "WBA15N_Tier", "WBA15S_Tier", "WBA16_Tier", "WBA17N_Tier", "WBA17S_Tier", "WBA18_Tier", "WBA19_Tier", "WBA20_Tier", "WBA21_Tier", "WBA22_Tier", "WBA23_Tier", "WBA24_Tier", "WBA25_Tier", "WBA26N_Tier", "WBA26S_Tier"]

SOD_gw = ["WBA50_Tier", "WBA60N_Tier", "WBA60S_Tier", "WBA61_Tier", "WBA62_Tier", "WBA63_Tier", "WBA64_Tier", "WBA71_Tier", "WBA72_Tier", "WBA73_Tier", "WBA90_Tier"]

Salinity_InDelta = ['Salinity_InDelta_Tier']

Salinity_Export = ['Salinity_Export_Tier']

In [ ]:
print(f"GW tiers available: {[c for c in gw_all.columns if '_Tier' in c][:5]}... ({len([c for c in gw_all.columns if '_Tier' in c])} total)")
print(f"Storage tiers available: {[c for c in storage_all.columns if '_Tier' in c]}")
print(f"Salinity tiers available: {[c for c in sal_all.columns if 'Tier' in c]}")

In [ ]:
nod_sod_df = pd.DataFrame(index=gw_all.index)
nod_sod_df.index.name = 'scenario'

nod_sod_df["NOD_GW_Mean"] = gw_all[NOD_gw].mean(axis=1)
nod_sod_df["SOD_GW_Mean"] = gw_all[SOD_gw].mean(axis=1)

nod_sod_df["NOD_Reservoir_Mean"] = storage_all[NOD_reservoirs].mean(axis=1).reindex(nod_sod_df.index)
nod_sod_df["SOD_Reservoir_Mean"] = storage_all[SOD_reservoirs].mean(axis=1).reindex(nod_sod_df.index)

nod_sod_df["Export_Salinity"] = sal_all["Salinity_Export_Tier"].reindex(nod_sod_df.index)
nod_sod_df["InDelta_Salinity"] = sal_all["Salinity_InDelta_Tier"].reindex(nod_sod_df.index)

nod_sod_df.reset_index(inplace=True)
nod_sod_df

In [ ]:
nod_sod_std_df = pd.DataFrame(index=gw_all.index)
nod_sod_std_df.index.name = 'scenario'

if len(NOD_gw) >= 2:
    nod_sod_std_df["NOD_GW_Std"] = gw_all[NOD_gw].std(axis=1)
else:
    nod_sod_std_df["NOD_GW_Std"] = 0
    
if len(SOD_gw) >= 2:
    nod_sod_std_df["SOD_GW_Std"] = gw_all[SOD_gw].std(axis=1)
else:
    nod_sod_std_df["SOD_GW_Std"] = 0
    
if len(NOD_reservoirs) >= 2:
    nod_sod_std_df["NOD_Reservoir_Std"] = storage_all[NOD_reservoirs].std(axis=1).reindex(nod_sod_std_df.index)
else:
    nod_sod_std_df["NOD_Reservoir_Std"] = 0
    
if len(SOD_reservoirs) >= 2:
    nod_sod_std_df["SOD_Reservoir_Std"] = storage_all[SOD_reservoirs].std(axis=1).reindex(nod_sod_std_df.index)
else:
    nod_sod_std_df["SOD_Reservoir_Std"] = 0
    
if len(Salinity_Export) >= 2:
    nod_sod_std_df["Export_Salinity_Std"] = sal_all[Salinity_Export].reindex(nod_sod_df.index)
else:
    nod_sod_std_df["Export_Salinity_Std"] = 0
    
if len(Salinity_InDelta) >= 2:
    nod_sod_std_df["InDelta_Salinity_Std"] = sal_all[Salinity_InDelta].reindex(nod_sod_df.index)
else:
    nod_sod_std_df["InDelta_Salinity_Std"] = 0

nod_sod_std_df.fillna(0, inplace = True)

nod_sod_std_df.reset_index(inplace=True)
nod_sod_std_df

In [ ]:
cols = ["NOD_GW_Mean", "SOD_GW_Mean", "NOD_Reservoir_Mean", "SOD_Reservoir_Mean", "Export_Salinity", "InDelta_Salinity"]
labels = cols
minmax = ["max", "max", "max", "max", "max", "max"]
ideal = "top"

for sset in SCENARIO_SETS:
    highlight_scenarios = sset['all_ids']
    highlight_indices = nod_sod_df.index[nod_sod_df["scenario"].isin(highlight_scenarios)].tolist()
    
    save_path = os.path.join(plots_base, f"parallel_NOD_SOD_{sset['name']}.png")
    
    fig, ax = custom_parallel_coordinates_highlight_scenarios(
        objs=nod_sod_df[cols],
        columns_axes=cols,
        axis_labels=labels,
        ideal_direction=ideal,
        minmaxs=minmax,
        highlight_indices=highlight_indices,
        highlight_colors=sset['colors'],
        highlight_descriptions=sset['labels'],
        title=f"Averaged NOD and SOD Tiers: {sset['name']}",
        fontsize=12,
        figsize=(22, 10),
        save_fig_filename=save_path)
    
    plt.show()
    print(f"Saved: {save_path}")

## 3. Tier-Based Radar Plots

In [ ]:
def plot_tier_radar_with_overlay_violins(df, cols, scenario_col, highlight_scenarios, highlight_colors, highlight_labels,
                                          title, std_df=None, max_tier=4, save_path=None, figsize=(12, 10)):
    """
    Radar plot with violin distributions overlaid along each radial axis.
    One violin per category showing distribution across ALL scenarios.
    Dot Sizes Adjusted to showcase Variance of Data
    """
    valid_scenarios = [s for s in highlight_scenarios if s in df[scenario_col].values]
    if len(valid_scenarios) < 1 or len(cols) < 3:
        print(f"Skip {title}: insufficient data")
        return
    
    highlight_mask = df[scenario_col].isin(valid_scenarios)
    highlight_data = df.loc[highlight_mask, cols]
    highlight_ids = df.loc[highlight_mask, scenario_col].values
    
    # Invert: Tier 1 -> 4 (outside), Tier 4 -> 1 (inside)
    invert = lambda x: (max_tier + 1) - x
    
    angles = np.linspace(0, 2*np.pi, len(cols), endpoint=False).tolist()
    
    fig, ax = plt.subplots(figsize=figsize, subplot_kw=dict(polar=True))
    
    # Draw ONE violin per category (all scenarios' values for that category)
    violin_width = 0.20
    
    for i, (col, angle) in enumerate(zip(cols, angles)):
        # Get ALL scenarios' values for this category
        all_values = df[col].dropna().values.astype(float)
        
        if len(all_values) < 3:
            continue
        
        inv_values = invert(all_values)
        unique_vals = np.unique(inv_values)
        
        if len(unique_vals) < 3:
            # Use simple range bar
            y_min, y_max = inv_values.min(), inv_values.max()
            width = violin_width * 0.5
            ax.fill([angle-width, angle+width, angle+width, angle-width],
                   [y_min, y_min, y_max, y_max], alpha=0.3, color='gray', edgecolor='gray')
        else:
            # Compute KDE for all scenarios
            try:
                kde = stats.gaussian_kde(inv_values, bw_method=0.4)
                y_range = np.linspace(0.5, max_tier + 0.5, 50)
                density = kde(y_range)
                density = density / density.max() * violin_width
                
                left_angles = angle - density
                right_angles = angle + density
                verts_theta = np.concatenate([left_angles, right_angles[::-1]])
                verts_r = np.concatenate([y_range, y_range[::-1]])
                
                ax.fill(verts_theta, verts_r, alpha=0.3, color='gray', edgecolor='darkgray', linewidth=0.5)
            except:
                pass

    use_std = std_df is not None
    dot_size_map = {}
    
    if use_std:
        for col in cols:
            std_col = col.replace('_Mean', '_Std')
    
            if std_col not in std_df.columns:
                continue
    
            std_vals = std_df[std_col].dropna().values.astype(float)
    
            if len(std_vals) < 3:
                continue

            max = 4
            min = 1
            max = (max-min)/2
    
            dot_size_map[col] = {
                "low": max/3,
                "mid": 2*max/3
            }


    
    # Draw highlighted scenario lines on top
    for idx, sc_id in zip(highlight_data.index, highlight_ids):
        if sc_id not in valid_scenarios:
            continue
    
        sc_idx = valid_scenarios.index(sc_id)
    
        inv_vals = invert(highlight_data.loc[idx]).values.tolist()
        angles_closed = angles + [angles[0]]
        inv_vals_closed = inv_vals + [inv_vals[0]]
    
        # draw line only
        ax.plot(
            angles_closed,
            inv_vals_closed,
            '-',
            lw=2.5,
            color=highlight_colors[sc_idx],
            label=highlight_labels[sc_idx],
            zorder=10
        )
    
        # draw markers one-by-one so size can vary per axis
        for col, angle, r in zip(cols, angles, inv_vals):
            std_col = col.replace('_Mean', '_Std')
            std_val = std_df.loc[idx, std_col] if (use_std and std_col in std_df.columns) else None

            thresholds = dot_size_map.get(col)

            if not use_std or thresholds is None or pd.isna(std_val):
                ms = 7
            elif std_val <= thresholds["low"]:
                ms = 5
            elif std_val <= thresholds["mid"]:
                ms = 10
            else:
                ms = 15
    
            ax.plot(
                angle,
                r,
                marker='o',
                color=highlight_colors[sc_idx],
                markersize=ms,
                markeredgecolor='black', 
                markeredgewidth=0.8,
                alpha=0.4,
                zorder=11
            )

    
    # Configure axes
    ax.set_xticks(angles)
    axis_labels = [c.replace('_Mean', '').replace('_', ' ') for c in cols]
    ax.set_xticklabels(axis_labels, fontsize=10)
    
    ax.set_ylim(0, max_tier + 0.5)
    ax.set_yticks([1, 2, 3, 4])
    ax.set_yticklabels(['Tier 4', 'Tier 3', 'Tier 2', 'Tier 1'], fontsize=9)

    # Create Color Legend
    scenario_legend = ax.legend(loc='upper right', bbox_to_anchor=(1.05, 1.05), fontsize=10)
    ax.set_title(title, fontsize=14, pad=20, fontweight='bold')

    # Create Dot Size Legend
    size_legend_elements = [
        Line2D([0], [0], marker='o', color='gray', label='Low variability',
               markersize=5, alpha=0.1, linestyle='None'),
        Line2D([0], [0], marker='o', color='gray', label='Medium variability',
               markersize=10, alpha=0.1, linestyle='None'),
        Line2D([0], [0], marker='o', color='gray', label='High variability',
               markersize=15, alpha=0.1, linestyle='None')
    ]
    
    size_legend = ax.legend(
        handles=size_legend_elements,
        title='Dot size (Std Dev)',
        loc='lower right',
        bbox_to_anchor=(1.05, 0.0),
        fontsize=9,
        title_fontsize=10
    )

    ax.add_artist(size_legend)
    ax.add_artist(scenario_legend)

    
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Saved: {save_path}")
    
    plt.show()
    return fig, ax

In [ ]:
# Generate tier radar plots with violin distributions (one violin per category, all scenarios)
radar_cols = ["NOD_GW_Mean", "SOD_GW_Mean", "NOD_Reservoir_Mean", 
              "SOD_Reservoir_Mean", "InDelta_Salinity", "Export_Salinity"]

for sset in SCENARIO_SETS:
    save_path = os.path.join(plots_base, f"tier_radar_violin_{sset['name']}.png")
    
    plot_tier_radar_with_overlay_violins(
        df=nod_sod_df,
        cols=radar_cols,
        scenario_col='scenario',
        highlight_scenarios=sset['all_ids'],
        highlight_colors=sset['colors'],
        highlight_labels=sset['labels'],
        title=f"Tier Radar: {sset['name']}",
        std_df = nod_sod_std_df,
        save_path=save_path
    )